In [ ]:

# KOMMENTAR: Denne celle importerer biblioteker og sætter device, float64 og SoftAdapt-only hyperparametre.
from pathlib import Path
import time, math
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# ============================================================
# Konfiguration
# ============================================================
USE_FLOAT64 = True

# KOMMENTAR: dtype styrer numerisk præcision; float64 er valgt for at reducere FD-cancellation.
dtype = torch.float64 if USE_FLOAT64 else torch.float32
torch.set_default_dtype(dtype)

# KOMMENTAR: Device vælges automatisk, så GPU bruges hvis CUDA er tilgængelig.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print('Using dtype :', dtype)

# KOMMENTAR: DATA_DIR bestemmer, hvor alle eksporterede .txt/.npy-filer læses fra eller gemmes.
DATA_DIR = Path('pinn_all_methods_exported_data')
DATA_DIR.mkdir(exist_ok=True)

# Problemparametre
v = 1.0   # diffusion i PDE'en u_t + c u_y - v u_yy = 0
c = 1.0   # advektion
Y = 2.0 * math.pi
T = 2.0

# Træningsparametre.
seeds = [1234, 2345, 3456, 4567, 5678]
N_f = 2000
N_b = 500
adam_epochs = 10000
adam_lr = 1e-3
resample_every = 100
eval_every_epochs = 100
lbfgs_max_iter = 500

# Tidsforsøg
max_time = 2 * 60
eval_every_time = 10

# Evalueringsgrid
N_x = 200
N_t = 200

RUN_EPOCH_EXPERIMENT = True
RUN_TIME_EXPERIMENT = True

method = 'softadapt'
label = 'SoftAdapt fixed LBFGS original ratios'


Using device: cuda
Using dtype : torch.float64


In [2]:

# KOMMENTAR: Denne celle opretter evalueringsgrid og hjælpefunktioner til skalering, sampling og eksport.
# ============================================================
# Grid, eksakt løsning og sampling
# ============================================================
x = np.linspace(0.0, Y, N_x)
t = np.linspace(0.0, T, N_t)
x_grid, t_grid = np.meshgrid(x, t)
x_tensor = torch.tensor(x_grid.reshape(-1, 1), dtype=dtype, device=device)
t_tensor = torch.tensor(t_grid.reshape(-1, 1), dtype=dtype, device=device)
U_exact = np.exp(-v * t_grid) * np.sin(x_grid - c * t_grid)

# KOMMENTAR: Gemmer arrays som tekstfiler, så plotting- og tabelkode kan køre separat fra træningen.
def save_array(name, arr):
    np.savetxt(DATA_DIR / f'{name}.txt', np.asarray(arr))

# KOMMENTAR: Skalerer fysiske koordinater til [-1,1], som er mere stabilt for tanh-netværk.
def scale_inputs(y, t):
    """Skalerer fysiske koordinater til [-1,1]^2 før netværket."""
    return 2.0*y/Y - 1.0, 2.0*t/T - 1.0

def uniform_sample_points():
    """Samme random sampling som i den oprindelige kode."""
    y_f = Y * torch.rand(N_f, 1, dtype=dtype, device=device)
    t_f = T * torch.rand(N_f, 1, dtype=dtype, device=device)
    t_b = T * torch.rand(N_b, 1, dtype=dtype, device=device)
    return y_f, t_f, t_b


In [3]:

# KOMMENTAR: Denne celle definerer baseline-netværket, som SoftAdapt-træningen bruger.
# ============================================================
# Model
# ============================================================
# KOMMENTAR: Standard fully-connected tanh-netværk brugt af baseline og flere metoder.
class MLP(nn.Module):
    def __init__(self, in_dim=2, hidden=64, depth=3, out_dim=1):
        super().__init__()
        layers = []
        last = in_dim
        for _ in range(depth):
            layers += [nn.Linear(last, hidden), nn.Tanh()]
            last = hidden
        layers.append(nn.Linear(last, out_dim))
        self.net = nn.Sequential(*layers)

        # Standard og stabil initialization til tanh-netværk.
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, z):
        return self.net(z)

# KOMMENTAR: Opretter den korrekte modeltype afhængigt af metodenavnet.
def make_model():
    return MLP().to(device=device, dtype=dtype)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# KOMMENTAR: Ensartet forward-funktion, så alle metoder bruger samme inputskalering.
def model_forward(model, y, t):
    ys, ts = scale_inputs(y, t)
    return model(torch.cat([ys, ts], dim=1))

_tmp_model = make_model()
print('SoftAdapt model parameters:', count_parameters(_tmp_model))
del _tmp_model


SoftAdapt model parameters: 8577


In [4]:

# KOMMENTAR: Denne celle definerer autograd-residual, boundary-loss og grid-evaluering.
# ============================================================
# PDE residual, boundary loss og evaluering
# ============================================================
# KOMMENTAR: Autograd-versionen beregner u_t, u_y og u_yy direkte via PyTorchs beregningsgraf.
def pde_residual_autograd(model, y, t):
    y = y.clone().detach().requires_grad_(True)
    t = t.clone().detach().requires_grad_(True)
    u = model_forward(model, y, t)
    u_t = torch.autograd.grad(u, t, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_y = torch.autograd.grad(u, y, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_yy = torch.autograd.grad(u_y, y, grad_outputs=torch.ones_like(u_y), create_graph=True)[0]
    r = u_t + c*u_y - v*u_yy
    return u, r

# KOMMENTAR: Håndhæver periodiske randbetingelser ved at matche både u og u_y ved y=0 og y=Y.
def periodic_boundary_loss(model, t_b):
    t_b = t_b.clone().detach().requires_grad_(True)
    y_left = torch.zeros_like(t_b, requires_grad=True)
    y_right = Y * torch.ones_like(t_b, requires_grad=True)

    u_left = model_forward(model, y_left, t_b)
    u_right = model_forward(model, y_right, t_b)

    u_left_y = torch.autograd.grad(u_left, y_left, grad_outputs=torch.ones_like(u_left), create_graph=True)[0]
    u_right_y = torch.autograd.grad(u_right, y_right, grad_outputs=torch.ones_like(u_right), create_graph=True)[0]

    return torch.mean((u_left-u_right)**2) + torch.mean((u_left_y-u_right_y)**2)

# KOMMENTAR: Evaluerer modellen mod den eksakte løsning på et fast grid til L2-, sup- og heatmap-data.
def evaluate_on_grid(model):
    model.eval()
    with torch.no_grad():
        u_pred = model_forward(model, x_tensor, t_tensor).detach().cpu().numpy()
    U_pred = u_pred.reshape(N_t, N_x)
    error = U_pred - U_exact
    abs_error = np.abs(error)
    rel_l2 = np.linalg.norm(error.ravel()) / np.linalg.norm(U_exact.ravel())
    sup_error = np.max(abs_error)
    model.train()
    return U_pred, abs_error, rel_l2, sup_error

# KOMMENTAR: Opdaterer samplingpunkter under træning; mesh-metoden bruger residualbaseret refinement.
def update_training_points(y_f, t_f, t_b, epoch):
    if epoch % resample_every == 0:
        return uniform_sample_points()
    return y_f, t_f, t_b


In [ ]:

# KOMMENTAR: Denne celle definerer SoftAdapt med originale ratioer samt en metode til at fryse vægte under LBFGS.
# ============================================================
# KOMMENTAR: SoftAdapt-klassen bruger originale ratioer og kan fryse vægte under LBFGS.
class StableSoftAdapt:

    def __init__(self, beta=5.0, eps=1e-8, enabled=True):
        self.beta = beta
        self.eps = eps
        self.enabled = enabled
        self.prev = None
        self.last_weights = None

    def _ones(self, names):
        return {name: torch.tensor(1.0, dtype=dtype, device=device) for name in names}

    def __call__(self, weighted_losses, update=True):
        names = list(weighted_losses.keys())
        if (not self.enabled) or self.prev is None:
            weights = self._ones(names)
        else:
            ratios = []
            for name in names:
                now = weighted_losses[name].detach()
                prev = self.prev[name]
                ratios.append(now / (prev + self.eps))
            ratios = torch.stack(ratios)

            # Original SoftAdapt-form: softmax(beta * relativ ændring) * antal komponenter.
            # Subtraktion af mean ændrer ikke softmax-sandsynlighederne, men forbedrer numerisk stabilitet.
            raw = torch.softmax(self.beta * (ratios - ratios.mean()), dim=0) * len(names)
            weights = {name: raw[j].detach() for j, name in enumerate(names)}

        if update:
            self.prev = {name: weighted_losses[name].detach() for name in names}
            self.last_weights = {name: weights[name].detach() for name in names}
        return weights

    def fixed_weights_for_lbfgs(self, names=('pde', 'ini', 'bc')):
        if self.last_weights is None:
            return self._ones(list(names))
        return {name: self.last_weights[name].detach().clone() for name in names}


# KOMMENTAR: Beregner alle loss-komponenter og kombinerer dem til den totale træningsloss.
def compute_components(model, y_batch, t_batch, t_b, softadapt=None, update_softadapt=False, fixed_weights=None):
    _, r = pde_residual_autograd(model, y_batch, t_batch)
    loss_pde = torch.mean(r**2)

    # Matcher den oprindelige kode: initialpunkterne bruger samme y_batch som PDE-punkterne.
    # Det holdes sådan her for at SoftAdapt-only-kørslen er sammenlignelig med de gamle filer.
    u_ini_pred = model_forward(model, y_batch, torch.zeros_like(y_batch))
    loss_ini = torch.mean((u_ini_pred - torch.sin(y_batch))**2)
    loss_bc = periodic_boundary_loss(model, t_b)

    weighted_losses = {
        'pde': loss_pde,
        'ini': 10.0 * loss_ini,
        'bc': 10.0 * loss_bc,
    }

    if fixed_weights is not None:
        # Under LBFGS bruges disse konstante vægte. SoftAdapt kaldes ikke inde i closure.
        weights = fixed_weights
    else:
        if softadapt is None:
            raise ValueError('SoftAdapt objekt mangler.')
        weights = softadapt(weighted_losses, update=update_softadapt)

    loss_total = (
        weights['pde'] * weighted_losses['pde']
        + weights['ini'] * weighted_losses['ini']
        + weights['bc']  * weighted_losses['bc']
    )
    return loss_total, loss_pde, loss_ini, loss_bc, weights


def loss_values(model, y_f, t_f, t_b):
    with torch.enable_grad():
        soft = StableSoftAdapt(enabled=False)
        _, pde, ini, bc, _ = compute_components(model, y_f, t_f, t_b, softadapt=soft, update_softadapt=False)
    return pde.detach().cpu().item(), ini.detach().cpu().item(), bc.detach().cpu().item()


In [6]:

# KOMMENTAR: Denne celle definerer LBFGS-polish, hvor SoftAdapt-vægtene holdes faste i closure-kaldene.
# ============================================================
# LBFGS med faste SoftAdapt-vægte
# ============================================================
# KOMMENTAR: Kører LBFGS som afsluttende polish efter Adam-fasen.
def run_lbfgs_polish(model, softadapt, y_f, t_f, t_b, verbose=True):
    # Som i den gamle kode: ny uniform batch til LBFGS-polish.
    y_f, t_f, t_b = uniform_sample_points()

    # VIGTIGT: Frys SoftAdapt-vægtene. De må ikke genberegnes inde i closure.
    fixed_weights = softadapt.fixed_weights_for_lbfgs()
    fixed_weights = {k: v.detach().to(device=device, dtype=dtype) for k, v in fixed_weights.items()}

    if verbose:
        _, _, rel_before, sup_before = evaluate_on_grid(model)
        pde0, ini0, bc0 = loss_values(model, y_f, t_f, t_b)
        print(
            'LBFGS before | '
            f'rel={rel_before:.10e} | sup={sup_before:.10e} | '
            f'pde={pde0:.3e} | ini={ini0:.3e} | bc={bc0:.3e} | '
            f'w=({fixed_weights["pde"].item():.3e}, {fixed_weights["ini"].item():.3e}, {fixed_weights["bc"].item():.3e})'
        )

    lbfgs = torch.optim.LBFGS(
        model.parameters(),
        lr=1.0,
        max_iter=lbfgs_max_iter,
        max_eval=lbfgs_max_iter * 2,
        tolerance_grad=1e-11,
        tolerance_change=1e-13,
        history_size=50,
        line_search_fn='strong_wolfe',
    )

    closure_calls = {'n': 0}
    def closure():
        closure_calls['n'] += 1
        lbfgs.zero_grad(set_to_none=True)
        loss_total, _, _, _, _ = compute_components(
            model, y_f, t_f, t_b,
            softadapt=softadapt,
            update_softadapt=False,
            fixed_weights=fixed_weights,
        )
        loss_total.backward()
        return loss_total

    t0 = time.perf_counter()
    lbfgs.step(closure)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    lbfgs_seconds = time.perf_counter() - t0

    if verbose:
        _, _, rel_after, sup_after = evaluate_on_grid(model)
        pde1, ini1, bc1 = loss_values(model, y_f, t_f, t_b)
        print(
            'LBFGS after  | '
            f'rel={rel_after:.10e} | sup={sup_after:.10e} | '
            f'pde={pde1:.3e} | ini={ini1:.3e} | bc={bc1:.3e} | '
            f'closure_calls={closure_calls["n"]} | seconds={lbfgs_seconds:.2f}'
        )

    return y_f, t_f, t_b, closure_calls['n'], lbfgs_seconds


In [7]:

# KOMMENTAR: Denne celle kører fixed-epoch SoftAdapt-eksperimentet og gemmer epoch-baserede filer.
# ============================================================
# Epoch experiment kun for SoftAdapt
# ============================================================
# KOMMENTAR: Kører fixed-epoch SoftAdapt-træningen over alle seeds og samler seedvise fejlkurver.
def train_epoch_experiment_softadapt():
    print('\n' + '='*80)
    print('Epoch experiment: SoftAdapt fixed LBFGS original ratios')
    print('='*80)

    epoch_eval_points = np.arange(0, adam_epochs + 1, eval_every_epochs, dtype=float)
    epoch_eval_points = np.append(epoch_eval_points, adam_epochs + lbfgs_max_iter)
    n_eval = len(epoch_eval_points)

    l2_errors = np.full((len(seeds), n_eval), np.nan)
    sup_errors = np.full((len(seeds), n_eval), np.nan)
    loss_pde_hist = np.full((len(seeds), n_eval), np.nan)
    loss_ini_hist = np.full((len(seeds), n_eval), np.nan)
    loss_bc_hist = np.full((len(seeds), n_eval), np.nan)
    weights_hist = np.full((len(seeds), n_eval, 3), np.nan)
    surface = np.full((N_t, N_x, len(seeds)), np.nan)
    pinn_solutions = np.full((N_t, N_x, len(seeds)), np.nan)

    for i, seed in enumerate(seeds):
        print(f'Training SoftAdapt fixed LBFGS original ratios with seed = {seed}')
        torch.manual_seed(seed)
        np.random.seed(seed)

        model = make_model()
        optimizer = torch.optim.Adam(model.parameters(), lr=adam_lr)
        softadapt = StableSoftAdapt(beta=5.0, enabled=True)
        y_f, t_f, t_b = uniform_sample_points()

        eval_idx = 0
        for epoch in range(adam_epochs + 1):
            if epoch % eval_every_epochs == 0:
                U_pred, abs_error, rel_l2, sup_error = evaluate_on_grid(model)
                l2_errors[i, eval_idx] = rel_l2
                sup_errors[i, eval_idx] = sup_error
                pde_v, ini_v, bc_v = loss_values(model, y_f, t_f, t_b)
                loss_pde_hist[i, eval_idx] = pde_v
                loss_ini_hist[i, eval_idx] = ini_v
                loss_bc_hist[i, eval_idx] = bc_v
                w = softadapt.fixed_weights_for_lbfgs()
                weights_hist[i, eval_idx, :] = [w['pde'].item(), w['ini'].item(), w['bc'].item()]
                eval_idx += 1

            if epoch == adam_epochs:
                break

            y_f, t_f, t_b = update_training_points(y_f, t_f, t_b, epoch)
            optimizer.zero_grad(set_to_none=True)
            loss_total, _, _, _, _ = compute_components(
                model, y_f, t_f, t_b,
                softadapt=softadapt,
                update_softadapt=True,
            )
            loss_total.backward()
            optimizer.step()

        # LBFGS efter Adam med faste SoftAdapt-vægte.
        y_f, t_f, t_b, n_closure, lbfgs_sec = run_lbfgs_polish(model, softadapt, y_f, t_f, t_b, verbose=True)

        U_pred, abs_error, rel_l2, sup_error = evaluate_on_grid(model)
        if eval_idx < n_eval:
            l2_errors[i, eval_idx] = rel_l2
            sup_errors[i, eval_idx] = sup_error
            pde_v, ini_v, bc_v = loss_values(model, y_f, t_f, t_b)
            loss_pde_hist[i, eval_idx] = pde_v
            loss_ini_hist[i, eval_idx] = ini_v
            loss_bc_hist[i, eval_idx] = bc_v
            w = softadapt.fixed_weights_for_lbfgs()
            weights_hist[i, eval_idx, :] = [w['pde'].item(), w['ini'].item(), w['bc'].item()]

        pinn_solutions[:, :, i] = U_pred
        surface[:, :, i] = abs_error

    # Samme filnavne som før for SoftAdapt, så plotting-koden kan genbruge dem.
    save_array('softadapt_x_grid', x_grid)
    save_array('softadapt_t_grid', t_grid)
    save_array('softadapt_epochs_eval', epoch_eval_points)
    save_array('softadapt_l2_error_vs_epochs', l2_errors)
    save_array('softadapt_infinity_norm_vs_epochs', sup_errors)
    save_array('softadapt_loss_pde_vs_epochs', loss_pde_hist)
    save_array('softadapt_loss_ini_vs_epochs', loss_ini_hist)
    save_array('softadapt_loss_bc_vs_epochs', loss_bc_hist)
    save_array('softadapt_enviroment_error', np.nanmean(surface, axis=2))
    save_array('softadapt_pinn_solution_mean', np.nanmean(pinn_solutions, axis=2))

    # Ekstra debug-fil. Plotting-koden behøver ikke bruge den.
    np.save(DATA_DIR / 'softadapt_weights_vs_epochs.npy', weights_hist)

    return l2_errors, sup_errors, weights_hist


In [8]:

# KOMMENTAR: Denne celle kører tidsbaseret SoftAdapt-eksperiment og gemmer time-baserede filer.
# ============================================================
# Time experiment kun for SoftAdapt
# ============================================================
# KOMMENTAR: Kører wall-clock-baseret SoftAdapt-træning med efterfølgende LBFGS-polish.
def train_time_experiment_softadapt():
    print('\n' + '='*80)
    print('Time experiment: SoftAdapt fixed LBFGS original ratios')
    print('='*80)

    eval_times = np.arange(0, max_time + 1, eval_every_time, dtype=float)
    n_time = len(eval_times)

    l2_errors_time = np.full((len(seeds), n_time), np.nan)
    sup_errors_time = np.full((len(seeds), n_time), np.nan)
    epochs_at_time = np.full((len(seeds), n_time), np.nan)
    actual_times = np.full((len(seeds), n_time), np.nan)

    l2_after_lbfgs = np.full(len(seeds), np.nan)
    sup_after_lbfgs = np.full(len(seeds), np.nan)
    actual_time_after_lbfgs = np.full(len(seeds), np.nan)
    epochs_before_lbfgs = np.full(len(seeds), np.nan)
    lbfgs_closure_calls = np.full(len(seeds), np.nan)
    lbfgs_seconds_arr = np.full(len(seeds), np.nan)

    for i, seed in enumerate(seeds):
        print(f'Training SoftAdapt fixed LBFGS original ratios with seed = {seed}')
        torch.manual_seed(seed)
        np.random.seed(seed)

        model = make_model()
        optimizer = torch.optim.Adam(model.parameters(), lr=adam_lr)
        softadapt = StableSoftAdapt(beta=5.0, enabled=True)
        y_f, t_f, t_b = uniform_sample_points()

        if device.type == 'cuda':
            torch.cuda.synchronize()
        start_time = time.perf_counter()
        epoch = 0
        time_idx = 0

        while time_idx < n_time:
            if device.type == 'cuda':
                torch.cuda.synchronize()
            elapsed = time.perf_counter() - start_time

            if elapsed >= eval_times[time_idx]:
                _, _, rel_l2, sup_error = evaluate_on_grid(model)
                l2_errors_time[i, time_idx] = rel_l2
                sup_errors_time[i, time_idx] = sup_error
                epochs_at_time[i, time_idx] = epoch
                actual_times[i, time_idx] = elapsed
                print(f'seed={seed} | target={eval_times[time_idx]:.0f}s | actual={elapsed:.1f}s | epoch={epoch} | rel L2={rel_l2:.6e}')
                time_idx += 1
                if time_idx >= n_time:
                    break

            y_f, t_f, t_b = update_training_points(y_f, t_f, t_b, epoch)
            optimizer.zero_grad(set_to_none=True)
            loss_total, _, _, _, _ = compute_components(
                model, y_f, t_f, t_b,
                softadapt=softadapt,
                update_softadapt=True,
            )
            loss_total.backward()
            optimizer.step()
            epoch += 1

        y_f, t_f, t_b, n_closure, lbfgs_sec = run_lbfgs_polish(model, softadapt, y_f, t_f, t_b, verbose=True)

        if device.type == 'cuda':
            torch.cuda.synchronize()
        elapsed_after = time.perf_counter() - start_time
        _, _, rel_l2, sup_error = evaluate_on_grid(model)

        l2_after_lbfgs[i] = rel_l2
        sup_after_lbfgs[i] = sup_error
        actual_time_after_lbfgs[i] = elapsed_after
        epochs_before_lbfgs[i] = epoch
        lbfgs_closure_calls[i] = n_closure
        lbfgs_seconds_arr[i] = lbfgs_sec

        print(
            f'seed={seed} | after LBFGS | actual={elapsed_after:.1f}s | '
            f'epochs_before_lbfgs={epoch} | rel L2={rel_l2:.10e} | '
            f'closure_calls={n_closure} | lbfgs_seconds={lbfgs_sec:.2f}'
        )

    save_array('softadapt_time', eval_times)
    save_array('softadapt_l2_error_time', l2_errors_time)
    save_array('softadapt_supremumsfejl_time', sup_errors_time)
    save_array('softadapt_epochs_at_time', epochs_at_time)
    save_array('softadapt_actual_times', actual_times)
    save_array('softadapt_l2_error_time_after_lbfgs', l2_after_lbfgs)
    save_array('softadapt_supremumsfejl_time_after_lbfgs', sup_after_lbfgs)
    save_array('softadapt_actual_time_after_lbfgs', actual_time_after_lbfgs)
    save_array('softadapt_epochs_before_time_lbfgs', epochs_before_lbfgs)

    # Ekstra debug-filer. Plotting-koden behøver ikke bruge dem.
    save_array('softadapt_lbfgs_closure_calls', lbfgs_closure_calls)
    save_array('softadapt_lbfgs_seconds', lbfgs_seconds_arr)

    return l2_errors_time, sup_errors_time, l2_after_lbfgs


In [ ]:

# KOMMENTAR: Denne celle starter SoftAdapt-only kørslerne og overskriver kun softadapt_*-filerne.
# ============================================================
# Kør kun SoftAdapt og overskriv kun softadapt_*.txt-filerne
# ============================================================
# KOMMENTAR: Flaget bestemmer, om epoch-eksperimentet køres.
if RUN_EPOCH_EXPERIMENT:
    softadapt_epoch_l2, softadapt_epoch_sup, softadapt_weights = train_epoch_experiment_softadapt()

# KOMMENTAR: Flaget bestemmer, om time-eksperimentet køres.
if RUN_TIME_EXPERIMENT:
    softadapt_time_l2, softadapt_time_sup, softadapt_time_after_lbfgs = train_time_experiment_softadapt()

print('Færdig. SoftAdapt-data er gemt i:', DATA_DIR.resolve())



Epoch experiment: SoftAdapt fixed LBFGS original ratios
Training SoftAdapt fixed LBFGS original ratios with seed = 1234


c:\Users\45304\torch_env\Lib\site-packages\torch\autograd\graph.py:865: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\cuda\CublasHandlePool.cpp:330.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


LBFGS before | rel=3.9037553471e-02 | sup=3.3505161283e-02 | pde=2.022e-03 | ini=3.309e-05 | bc=1.953e-05 | w=(1.002e+00, 9.952e-01, 1.003e+00)
LBFGS after  | rel=9.1694516498e-04 | sup=1.2493446044e-03 | pde=1.050e-05 | ini=1.451e-07 | bc=9.057e-08 | closure_calls=534 | seconds=13.29
Training SoftAdapt fixed LBFGS original ratios with seed = 2345
LBFGS before | rel=7.6984743946e-03 | sup=8.2396874264e-03 | pde=4.805e-04 | ini=1.637e-06 | bc=3.424e-06 | w=(1.002e+00, 9.963e-01, 1.002e+00)
LBFGS after  | rel=8.2866730700e-04 | sup=1.4006621661e-03 | pde=1.035e-05 | ini=1.202e-07 | bc=2.196e-07 | closure_calls=533 | seconds=12.61
Training SoftAdapt fixed LBFGS original ratios with seed = 3456
LBFGS before | rel=5.8797304649e-03 | sup=5.3593777241e-03 | pde=2.650e-04 | ini=2.460e-06 | bc=4.315e-06 | w=(1.001e+00, 9.998e-01, 9.989e-01)
LBFGS after  | rel=7.0751085322e-04 | sup=8.3747820780e-04 | pde=8.136e-06 | ini=3.218e-08 | bc=1.475e-07 | closure_calls=538 | seconds=12.66
Training SoftA